# Graph Attention Networks (GAT) — Colab

This notebook mirrors `code/model.py`, `code/train.py`, and `code/evaluate.py` in the repo: **2-layer GAT** on **Cora** / **CiteSeer** with 100 runs and **mean ± std** test accuracy (paper Table 2 targets: **83.0 ± 0.7%** / **72.5 ± 0.7%**).

## Colab setup

1. **Runtime → Change runtime type →** enable **GPU** (recommended; 100 full runs on CPU is slow).
2. Run the **Install** cell below once per runtime.
3. Set `NUM_RUNS` in the last section (use `3` for a quick smoke test, `100` to match the paper’s protocol).
4. Optional: mount Drive and set `PLANETOID_PARENT` / `RESULTS_DIR` to a folder under `/content/drive/MyDrive/...` so data and CSVs survive disconnects.

In [ ]:
# @title Install dependencies (run once per Colab session)
!pip install -q torch-geometric

In [ ]:
# @title Config: paths and imports
import csv
import pathlib
import sys
from typing import List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv
from torch_geometric.transforms import NormalizeFeatures

# Planetoid download/cache. Under /content so it persists for the Colab session.
PLANETOID_PARENT = pathlib.Path("/content/gat_data")
RESULTS_DIR = pathlib.Path("/content/gat_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| cwd:", pathlib.Path().resolve())

In [ ]:
# @title Model (same as `code/model.py`)
class GAT(nn.Module):
    def __init__(self, num_features: int, num_classes: int, dropout: float = 0.6):
        super().__init__()
        self.dropout = dropout

        self.conv1 = GATConv(
            in_channels=num_features,
            out_channels=8,
            heads=8,
            dropout=dropout,
            concat=True,
        )
        self.conv2 = GATConv(
            in_channels=8 * 8,
            out_channels=num_classes,
            heads=1,
            dropout=dropout,
            concat=False,
        )

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

In [ ]:
# @title Training (same as `code/train.py`)
LR = 0.005
WEIGHT_DECAY = 5e-4
DROPOUT = 0.6
EPOCHS = 10_000
PATIENCE = 100


def train_epoch(model, data, optimizer):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def eval_split(model, data):
    model.eval()
    out = model(data.x, data.edge_index)
    val_loss = F.nll_loss(out[data.val_mask], data.y[data.val_mask]).item()
    pred = out.argmax(dim=1)
    val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
    test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
    return val_loss, val_acc, test_acc


def run(
    dataset_name: str,
    seed: int = 42,
    data_parent: Optional[pathlib.Path] = None,
) -> float:
    torch.manual_seed(seed)
    parent = data_parent or PLANETOID_PARENT
    parent.mkdir(parents=True, exist_ok=True)
    root = str(parent / dataset_name)

    dataset = Planetoid(
        root=root,
        name=dataset_name,
        transform=NormalizeFeatures(),
    )
    data = dataset[0]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data = data.to(device)

    model = GAT(
        num_features=dataset.num_features,
        num_classes=dataset.num_classes,
        dropout=DROPOUT,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val_loss = float("inf")
    best_test_acc = 0.0
    patience_counter = 0

    for _epoch in range(1, EPOCHS + 1):
        train_epoch(model, data, optimizer)
        val_loss, _val_acc, test_acc = eval_split(model, data)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_test_acc = test_acc
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break

    return best_test_acc

In [ ]:
# @title Multi-run evaluation (same as `code/evaluate.py`)
def run_many(dataset_name: str, num_runs: int) -> Tuple[float, float, pathlib.Path]:
    csv_path = RESULTS_DIR / f"{dataset_name}_results.csv"
    accuracies: List[float] = []
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["run", "test_accuracy"])
        for i in range(1, num_runs + 1):
            acc = run(dataset_name, seed=i)
            accuracies.append(acc)
            w.writerow([i, f"{acc:.6f}"])
            f.flush()
            sys.stdout.flush()
            print(f"Run {i:3d}/{num_runs}  test_acc={acc * 100:.2f}%", flush=True)
    mean_acc = float(np.mean(accuracies) * 100)
    std_acc = float(np.std(accuracies) * 100)
    print("\n" + "=" * 45)
    print(f"Dataset : {dataset_name}")
    print(f"Runs    : {num_runs}")
    print(f"Mean    : {mean_acc:.2f}%")
    print(f"Std     : {std_acc:.2f}%")
    print(f"Results saved to: {csv_path}")
    print("=" * 45)
    return mean_acc, std_acc, csv_path

In [ ]:
# @title Run experiments
# Set NUM_RUNS=100 to match the paper; use 3–5 for a quick test.
NUM_RUNS = 100
DATASETS = ("Cora", "CiteSeer")
summary = {}
for name in DATASETS:
    m, s, _p = run_many(name, num_runs=NUM_RUNS)
    summary[name] = (m, s)
print("\nSummary (mean, std) %:")
for k, (m, s) in summary.items():
    print(f"  {k}: {m:.2f} ± {s:.2f}")